In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp01
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)




# ════════════════════════════════════════════════════════════════════════
# MLFP01 — Exercise 2: Filtering and Transforming Data
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   After completing this exercise, you will be able to:
#   - Filter rows using Boolean logic with comparison operators (>, <, ==, !=)
#   - Combine filters with & (AND), | (OR), and ~ (NOT)
#   - Select, rename, and reorder columns to focus your analysis
#   - Create new derived columns with with_columns() and alias()
#   - Apply conditional column logic with pl.when().then().otherwise()
#   - Chain multiple Polars operations together in a readable pipeline
#
# PREREQUISITES: Complete Exercise 1 first (variables, DataFrames, describe()).
#
# ESTIMATED TIME: ~150-180 min
#
# TASKS:
#   1.  Inspect the HDB dataset and understand its structure
#   2.  Single-condition filters — one rule at a time
#   3.  Compound filters — combining multiple conditions with & and |
#   4.  Negation and set membership — ~ and .is_in()
#   5.  Select, rename, and reorder columns
#   6.  Create derived columns with with_columns() and alias()
#   7.  Date parsing and extraction — string to date, extract year/month
#   8.  Conditional columns with pl.when().then().otherwise()
#   9.  Sorting — single-key, multi-key, and descending
#   10. Method chaining — building analysis pipelines step by step
#
# DATASET: Singapore HDB resale flat transactions
#   Source: Housing & Development Board (data.gov.sg)
#   Rows: ~500,000 transactions | Columns: month, town, flat_type,
#   floor_area_sqm, resale_price, and more
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import polars as pl



### Data Loading


In [ ]:
loader = MLFPDataLoader()
hdb = loader.load("mlfp01", "hdb_resale.parquet")

print("=" * 60)
print("  MLFP01 Exercise 2: Filtering and Transforming Data")
print("=" * 60)
print(f"\n  Data loaded: hdb_resale.parquet ({hdb.height:,} rows, {hdb.width} columns)")
print(f"  You're ready to start!\n")

print("=== HDB Resale Dataset ===")
print(f"Shape: {hdb.shape}")
print(f"Columns: {hdb.columns}")
print(hdb.head(3))



## TASK 1: Inspect the HDB dataset — understand before you filter


In [ ]:
# Before writing any filter, understand what values exist in each column.
# Otherwise you'll filter for "Ang Mo Kio" when the data says "ANG MO KIO"
# and get zero rows back — a common beginner mistake.

# --- 1a: Data types and null counts ---
print("\n=== Column Details ===")
for col_name, dtype in zip(hdb.columns, hdb.dtypes):
    null_count = hdb[col_name].null_count()
    null_pct = null_count / hdb.height * 100
    print(f"  {col_name:>25}: {str(dtype):<12} nulls={null_count:,} ({null_pct:.1f}%)")

# --- 1b: Unique values for categorical columns ---
# .unique() returns all distinct values in a column
# .n_unique() counts them without listing — faster for large datasets
towns = hdb["town"].unique().sort()
flat_types = hdb["flat_type"].unique().sort()

print(f"\n=== Unique Towns ({towns.len()}) ===")
print(f"  {towns.to_list()}")
print(f"\n=== Flat Types ({flat_types.len()}) ===")
print(f"  {flat_types.to_list()}")

# --- 1c: Price distribution overview ---
print(f"\n=== Price Overview ===")
print(f"  Min:    S${hdb['resale_price'].min():>12,.0f}")
print(f"  Q1:     S${hdb['resale_price'].quantile(0.25):>12,.0f}")
print(f"  Median: S${hdb['resale_price'].quantile(0.50):>12,.0f}")
print(f"  Q3:     S${hdb['resale_price'].quantile(0.75):>12,.0f}")
print(f"  Max:    S${hdb['resale_price'].max():>12,.0f}")

# --- 1d: Date range ---
date_min = hdb["month"].min()
date_max = hdb["month"].max()
print(f"\n=== Date Range ===")
print(f"  Earliest: {date_min}")
print(f"  Latest:   {date_max}")
# INTERPRETATION: Understanding the data range is essential for filtering.
# If the earliest date is 2012-01 and you filter for 2010, you'll get zero
# rows and think your code is broken. Always check ranges first.



### Checkpoint 1


In [ ]:
assert towns.len() > 0, "Should have at least one town"
assert flat_types.len() > 0, "Should have at least one flat type"
assert hdb["resale_price"].min() > 0, "Min price should be positive"
assert date_min is not None, "Date min should not be None"
print("\n✓ Checkpoint 1 passed — dataset inspected, ranges understood\n")



## TASK 2: Single-condition filters — one rule at a time


In [ ]:
# A Boolean is True or False. Polars lets you write conditions using
# pl.col("column_name") and comparison operators: ==, !=, >, <, >=, <=
# .filter() keeps rows where the expression evaluates to True.

# --- 2a: Filter by exact string match ---
ang_mo_kio = hdb.filter(pl.col("town") == "ANG MO KIO")
print(f"\nAng Mo Kio transactions: {ang_mo_kio.height:,}")
print(f"  ({ang_mo_kio.height / hdb.height:.1%} of all transactions)")

# --- 2b: Filter by numeric comparison ---
expensive = hdb.filter(pl.col("resale_price") > 1_000_000)
print(f"Million-dollar HDBs (>S$1M): {expensive.height:,}")
print(f"  Most common town: {expensive['town'].mode()[0]}")
# mode() returns the most frequent value — the statistical mode

# --- 2c: Filter by price range ---
affordable = hdb.filter(
    (pl.col("resale_price") >= 300_000) & (pl.col("resale_price") <= 500_000)
)
print(f"Transactions S$300k-500k: {affordable.height:,}")

# --- 2d: Filter by flat type ---
four_room = hdb.filter(pl.col("flat_type") == "4 ROOM")
print(f"4-room flats: {four_room.height:,}")

# --- 2e: Filter by floor area ---
large_flats = hdb.filter(pl.col("floor_area_sqm") >= 120)
print(f"Large flats (>=120 sqm): {large_flats.height:,}")
print(f"  Avg price: S${large_flats['resale_price'].mean():,.0f}")

# --- 2f: String-based filter with .str accessor ---
# Polars has a .str accessor for string operations inside expressions
# .str.contains() checks if a substring exists anywhere in the string
# This is useful when exact match is too strict
mature_estates = hdb.filter(pl.col("town").str.contains("QUEENSTOWN|BISHAN|TOA PAYOH"))
print(f"Mature estates (Queenstown/Bishan/Toa Payoh): {mature_estates.height:,}")



### Checkpoint 2


In [ ]:
assert ang_mo_kio.height > 0, "AMK filter returned no rows — check column values"
assert affordable.height < hdb.height, "Affordable filter should reduce row count"
assert four_room.height > 0, "4-room filter should return rows"
assert large_flats.height < hdb.height, "Large flat filter should reduce count"
assert mature_estates.height > 0, "Mature estates filter should return rows"
print("\n✓ Checkpoint 2 passed — single-condition filters working\n")



## TASK 3: Compound filters — combining multiple conditions


In [ ]:
# & means AND — BOTH conditions must be True
# | means OR — EITHER condition can be True
# You MUST wrap each condition in parentheses when combining.
# Without parentheses, Python's operator precedence gives wrong results.

# --- 3a: AND — all conditions must be True ---
# Business question: "How many 4-room flats in Ang Mo Kio sold under S$500k?"
amk_4room_affordable = hdb.filter(
    (pl.col("town") == "ANG MO KIO")
    & (pl.col("flat_type") == "4 ROOM")
    & (pl.col("resale_price") <= 500_000)
)
print(f"AMK 4-room under S$500k: {amk_4room_affordable.height:,}")

# --- 3b: OR — either condition can be True ---
# Business question: "Which transactions are either very cheap or very expensive?"
extreme_prices = hdb.filter(
    (pl.col("resale_price") < 200_000) | (pl.col("resale_price") > 900_000)
)
print(f"Extreme prices (<200k or >900k): {extreme_prices.height:,}")

# --- 3c: AND + OR combined ---
# Business question: "In central towns, which are either very large or very expensive?"
central_towns = ["BISHAN", "TOA PAYOH", "QUEENSTOWN", "BUKIT MERAH"]
central_special = hdb.filter(
    pl.col("town").is_in(central_towns)
    & ((pl.col("floor_area_sqm") >= 130) | (pl.col("resale_price") >= 800_000))
)
print(f"Central large-or-expensive: {central_special.height:,}")

# --- 3d: Progressive filtering (chaining .filter() calls) ---
# You can also chain multiple .filter() calls — equivalent to AND.
# This is more readable when you have many conditions.
progressive = (
    hdb.filter(pl.col("town") == "BISHAN")
    .filter(pl.col("flat_type").is_in(["4 ROOM", "5 ROOM"]))
    .filter(pl.col("resale_price") >= 400_000)
    .filter(pl.col("resale_price") <= 700_000)
)
print(f"Bishan 4/5-room S$400k-700k: {progressive.height:,}")
# INTERPRETATION: Compound filters answer specific business questions.
# A property investor asking "What can I get in Bishan for S$400-700k?"
# would use exactly this filter. The count tells them how liquid the
# market is — more transactions = easier to buy and sell.



### Checkpoint 3


In [ ]:
assert (
    amk_4room_affordable.height <= ang_mo_kio.height
), "Combined filter should be a subset of single-town filter"
assert extreme_prices.height > 0, "Should have some extreme-price transactions"
assert central_special.height > 0, "Central special filter should return rows"
assert progressive.height > 0, "Progressive filter should return rows"
print("\n✓ Checkpoint 3 passed — compound filters working correctly\n")



## TASK 4: Negation and set membership — ~ and .is_in()


In [ ]:
# --- 4a: Negation with ~ ---
# ~ inverts a Boolean expression: True becomes False and vice versa.
# "Everything that is NOT Ang Mo Kio"
not_amk = hdb.filter(~(pl.col("town") == "ANG MO KIO"))
print(f"Not Ang Mo Kio: {not_amk.height:,}")
assert (
    not_amk.height + ang_mo_kio.height == hdb.height
), "AMK + not-AMK should equal total"
print(f"  Check: {ang_mo_kio.height:,} + {not_amk.height:,} = {hdb.height:,} ✓")

# --- 4b: .is_in() — match against a list of values ---
# Much cleaner than chaining (col == "A") | (col == "B") | (col == "C")
central_towns_list = ["BISHAN", "TOA PAYOH", "QUEENSTOWN", "BUKIT MERAH", "BUKIT TIMAH"]
central = hdb.filter(pl.col("town").is_in(central_towns_list))
print(
    f"\nCentral towns ({len(central_towns_list)} towns): {central.height:,} transactions"
)

# --- 4c: NOT in a set — combining ~ with .is_in() ---
peripheral = hdb.filter(~pl.col("town").is_in(central_towns_list))
print(f"Peripheral towns: {peripheral.height:,} transactions")
assert (
    central.height + peripheral.height == hdb.height
), "Central + peripheral should equal total"

# --- 4d: .is_between() — range filter shorthand ---
# .is_between() is cleaner than (col >= low) & (col <= high)
mid_range = hdb.filter(pl.col("resale_price").is_between(400_000, 600_000))
print(f"\nS$400k-600k (is_between): {mid_range.height:,}")

# --- 4e: .is_null() and .is_not_null() ---
# Check for missing values — critical before any analysis
for col_name in hdb.columns:
    nc = hdb[col_name].null_count()
    if nc > 0:
        print(f"  {col_name}: {nc:,} nulls")

# Filter rows where a specific column is NOT null
non_null_rows = hdb.filter(pl.col("resale_price").is_not_null())
print(f"Rows with non-null price: {non_null_rows.height:,}")

# INTERPRETATION: .is_in() is the Polars equivalent of SQL's IN clause.
# It's faster and more readable than chaining OR conditions. For property
# analysis, you'll often define a list of "comparable" towns and filter
# with .is_in() to create a peer group comparison.



### Checkpoint 4


In [ ]:
assert not_amk.height > 0, "Negation filter should return rows"
assert central.height > 0, "is_in filter should return rows"
assert central.height + peripheral.height == hdb.height, "Central + peripheral = total"
assert mid_range.height > 0, "is_between should return rows"
print("\n✓ Checkpoint 4 passed — negation, set membership, and null checks working\n")



## TASK 5: Select, rename, and reorder columns


In [ ]:
# --- 5a: .select() — keep only the columns you need ---
# This is important: always work with the smallest DataFrame that answers
# your question. Extra columns waste memory and clutter your output.
core_cols = hdb.select(
    "month",
    "town",
    "flat_type",
    "floor_area_sqm",
    "resale_price",
)
print(f"\nAfter select: {core_cols.columns}")
print(f"Columns: {core_cols.width} (was {hdb.width})")

# --- 5b: .select() with expressions — compute during selection ---
# You can select existing columns AND computed expressions in one call
summary_view = hdb.select(
    "town",
    "flat_type",
    "resale_price",
    (pl.col("resale_price") / pl.col("floor_area_sqm")).alias("price_per_sqm"),
)
print(f"\nSummary view columns: {summary_view.columns}")
print(summary_view.head(3))

# --- 5c: .rename() — change column names ---
renamed = core_cols.rename(
    {
        "month": "sale_month",
        "floor_area_sqm": "area_sqm",
        "resale_price": "price",
    }
)
print(f"\nAfter rename: {renamed.columns}")
print(renamed.head(3))

# --- 5d: Column ordering ---
# .select() with an explicit list also reorders columns
reordered = hdb.select(
    "resale_price",  # Price first — the variable we care about most
    "town",
    "flat_type",
    "floor_area_sqm",
    "month",
)
print(f"\nReordered columns: {reordered.columns}")

# --- 5e: .drop() — remove specific columns ---
# The opposite of .select() — keep everything EXCEPT the named columns
dropped = hdb.drop("block", "street_name")
print(f"\nAfter dropping block, street_name: {dropped.width} columns (was {hdb.width})")



### Checkpoint 5


In [ ]:
assert core_cols.width == 5, "core_cols should have exactly 5 columns"
assert "resale_price" not in renamed.columns, "resale_price should be renamed to price"
assert "price" in renamed.columns, "renamed DataFrame should have a 'price' column"
assert "price_per_sqm" in summary_view.columns, "summary_view should have price_per_sqm"
assert reordered.columns[0] == "resale_price", "First column should be resale_price"
print("\n✓ Checkpoint 5 passed — select, rename, reorder, and drop working\n")



## TASK 6: Derived columns with with_columns() and alias()


In [ ]:
# .with_columns() adds new columns (or replaces existing ones) without
# removing any other columns. This is how you engineer features.
# .alias() gives the new column a name.

# --- 6a: Price per square metre ---
hdb = hdb.with_columns(
    (pl.col("resale_price") / pl.col("floor_area_sqm")).alias("price_per_sqm"),
)
print(f"\n=== Price per sqm ===")
print(f"  Mean:   S${hdb['price_per_sqm'].mean():,.0f}/sqm")
print(f"  Median: S${hdb['price_per_sqm'].median():,.0f}/sqm")
# INTERPRETATION: price_per_sqm normalises for flat size — a 3-room flat in
# Bishan may cost less than a 5-room flat in Jurong, but price_per_sqm
# tells you which neighbourhood is truly more expensive per unit area.

# --- 6b: Multiple columns in one call ---
hdb = hdb.with_columns(
    # Parse "month" string (e.g. "2023-01") into a proper date
    pl.col("month").str.to_date("%Y-%m").alias("transaction_date"),
    # Extract year as integer
    pl.col("month").str.slice(0, 4).cast(pl.Int32).alias("year"),
    # Extract month number
    pl.col("month").str.slice(5, 2).cast(pl.Int32).alias("month_num"),
)

# --- 6c: Floor area categories ---
hdb = hdb.with_columns(
    pl.when(pl.col("floor_area_sqm") < 70)
    .then(pl.lit("small"))
    .when(pl.col("floor_area_sqm") < 100)
    .then(pl.lit("medium"))
    .when(pl.col("floor_area_sqm") < 130)
    .then(pl.lit("large"))
    .otherwise(pl.lit("executive"))
    .alias("size_category"),
)

# --- 6d: Price per sqm quartile flag ---
# Classify each transaction relative to the overall market
psm_q75 = hdb["price_per_sqm"].quantile(0.75)
psm_q25 = hdb["price_per_sqm"].quantile(0.25)

hdb = hdb.with_columns(
    pl.when(pl.col("price_per_sqm") >= psm_q75)
    .then(pl.lit("premium"))
    .when(pl.col("price_per_sqm") <= psm_q25)
    .then(pl.lit("value"))
    .otherwise(pl.lit("mainstream"))
    .alias("market_segment"),
)

print(f"\n=== After adding derived columns ===")
new_cols = [
    "price_per_sqm",
    "transaction_date",
    "year",
    "month_num",
    "size_category",
    "market_segment",
]
print(hdb.select(new_cols).head(5))



### Checkpoint 6


In [ ]:
assert "price_per_sqm" in hdb.columns, "price_per_sqm column should be added"
assert "transaction_date" in hdb.columns, "transaction_date column should be added"
assert "year" in hdb.columns, "year column should be added"
assert "size_category" in hdb.columns, "size_category column should be added"
assert "market_segment" in hdb.columns, "market_segment column should be added"
sample_psm = hdb["price_per_sqm"].drop_nulls()[0]
assert sample_psm > 0, "price_per_sqm should be positive"
print("\n✓ Checkpoint 6 passed — derived columns created correctly\n")



## TASK 7: Date parsing and extraction


In [ ]:
# Dates are crucial for time-series analysis. Polars stores dates as
# actual Date types, not strings — this enables arithmetic and extraction.

# --- 7a: Date arithmetic ---
# How many days between the earliest and latest transaction?
earliest = hdb["transaction_date"].min()
latest = hdb["transaction_date"].max()
date_span = (latest - earliest).days

print(f"=== Date Analysis ===")
print(f"Earliest transaction: {earliest}")
print(f"Latest transaction:   {latest}")
print(f"Span: {date_span:,} days ({date_span / 365.25:.1f} years)")

# --- 7b: Temporal distribution ---
# How many transactions per year?
yearly_counts = hdb.group_by("year").agg(pl.len().alias("count")).sort("year")
print(f"\n=== Transactions per Year ===")
for row in yearly_counts.iter_rows(named=True):
    bar = "█" * (row["count"] // 2000)
    print(f"  {row['year']}: {row['count']:>7,} {bar}")

# --- 7c: Monthly seasonality ---
monthly_counts = (
    hdb.group_by("month_num").agg(pl.len().alias("count")).sort("month_num")
)
print(f"\n=== Transactions by Month (all years) ===")
month_names = {
    1: "Jan",
    2: "Feb",
    3: "Mar",
    4: "Apr",
    5: "May",
    6: "Jun",
    7: "Jul",
    8: "Aug",
    9: "Sep",
    10: "Oct",
    11: "Nov",
    12: "Dec",
}
for row in monthly_counts.iter_rows(named=True):
    name = month_names.get(row["month_num"], "?")
    bar = "█" * (row["count"] // 1000)
    print(f"  {name}: {row['count']:>7,} {bar}")
# INTERPRETATION: Transaction volume varies by month. Singapore's property
# market typically sees dips around Chinese New Year (Jan/Feb) and year-end
# (Dec). Higher volume months have more data points, making their price
# statistics more reliable.

# --- 7d: Extract quarter ---
hdb = hdb.with_columns(
    ((pl.col("month_num") - 1) // 3 + 1).alias("quarter"),
)
quarterly = (
    hdb.group_by("year", "quarter")
    .agg(
        pl.len().alias("transactions"),
        pl.col("resale_price").median().alias("median_price"),
    )
    .sort("year", "quarter")
)
print(f"\n=== Quarterly Summary (last 8 quarters) ===")
print(quarterly.tail(8))



### Checkpoint 7


In [ ]:
assert date_span > 0, "Date span should be positive"
assert yearly_counts.height > 0, "Should have yearly counts"
assert monthly_counts.height == 12, "Should have 12 monthly counts"
assert "quarter" in hdb.columns, "quarter column should exist"
print("\n✓ Checkpoint 7 passed — date parsing and temporal analysis working\n")



## TASK 8: Conditional columns with pl.when().then().otherwise()


In [ ]:
# pl.when() is Polars' if/else for column creation.
# You can chain .when().then() as many times as you need.

# --- 8a: Price tier classification ---
hdb = hdb.with_columns(
    pl.when(pl.col("resale_price") < 350_000)
    .then(pl.lit("budget"))
    .when(pl.col("resale_price") < 500_000)
    .then(pl.lit("mid_range"))
    .when(pl.col("resale_price") < 700_000)
    .then(pl.lit("premium"))
    .otherwise(pl.lit("luxury"))
    .alias("price_tier")
)

tier_counts = (
    hdb.group_by("price_tier")
    .agg(
        pl.len().alias("count"),
        pl.col("resale_price").median().alias("median_price"),
        pl.col("floor_area_sqm").median().alias("median_area"),
    )
    .sort("median_price")
)
print(f"\n=== Price Tier Distribution ===")
for row in tier_counts.iter_rows(named=True):
    pct = row["count"] / hdb.height * 100
    bar = "█" * int(pct)
    print(
        f"  {row['price_tier']:<12} {row['count']:>8,} ({pct:5.1f}%) "
        f"median=S${row['median_price']:>10,.0f}  area={row['median_area']:.0f}sqm  {bar}"
    )

# --- 8b: Flat age proxy ---
# HDB leases are 99 years. The transaction date minus the flat's approximate
# build year gives remaining lease. We don't have build year directly, but
# we can estimate from the lease commencement date if available.
# For now, classify by era based on first transaction year per town+block.
hdb = hdb.with_columns(
    pl.when(pl.col("year") <= 2015)
    .then(pl.lit("pre-2016"))
    .when(pl.col("year") <= 2019)
    .then(pl.lit("2016-2019"))
    .when(pl.col("year") <= 2022)
    .then(pl.lit("2020-2022"))
    .otherwise(pl.lit("2023+"))
    .alias("transaction_era")
)

era_summary = (
    hdb.group_by("transaction_era")
    .agg(
        pl.len().alias("count"),
        pl.col("resale_price").median().alias("median_price"),
        pl.col("price_per_sqm").median().alias("median_psm"),
    )
    .sort("transaction_era")
)
print(f"\n=== Price by Transaction Era ===")
print(era_summary)
# INTERPRETATION: Comparing median prices across eras reveals price inflation.
# If 2023+ median is S$150k above pre-2016 median, that's the cumulative
# price appreciation. Dividing by the years gives a rough annual growth rate.

# --- 8c: Boolean flag columns ---
hdb = hdb.with_columns(
    (pl.col("resale_price") >= 1_000_000).alias("is_million_dollar"),
    (pl.col("floor_area_sqm") >= 100).alias("is_large_flat"),
)

million_count = hdb.filter(pl.col("is_million_dollar")).height
large_count = hdb.filter(pl.col("is_large_flat")).height
print(f"\nMillion-dollar HDBs: {million_count:,} ({million_count / hdb.height:.1%})")
print(f"Large flats (>=100sqm): {large_count:,} ({large_count / hdb.height:.1%})")



### Checkpoint 8


In [ ]:
assert "price_tier" in hdb.columns, "price_tier column should be added"
tier_values = set(hdb["price_tier"].unique().to_list())
expected_tiers = {"budget", "mid_range", "premium", "luxury"}
assert (
    tier_values == expected_tiers
), f"Expected tiers {expected_tiers}, got {tier_values}"
assert "transaction_era" in hdb.columns, "transaction_era should exist"
assert "is_million_dollar" in hdb.columns, "is_million_dollar should exist"
print("\n✓ Checkpoint 8 passed — conditional columns created with all tiers\n")



## TASK 9: Sorting — single-key, multi-key, and stable sorts


In [ ]:
# --- 9a: Single-key sort ---
by_price_desc = hdb.sort("resale_price", descending=True)
print(f"=== Most Expensive Transactions ===")
print(
    by_price_desc.select(
        "town", "flat_type", "resale_price", "floor_area_sqm", "month"
    ).head(5)
)

by_price_asc = hdb.sort("resale_price")
print(f"\n=== Cheapest Transactions ===")
print(
    by_price_asc.select(
        "town", "flat_type", "resale_price", "floor_area_sqm", "month"
    ).head(5)
)

# --- 9b: Multi-key sort ---
# First sort by town (alphabetically), then by price within each town
by_town_price = hdb.sort("town", "resale_price", descending=[False, True])
print(f"\n=== Sorted by Town then Price (desc) ===")
# Show the most expensive flat in each of the first 3 towns
seen_towns: set[str] = set()
for row in by_town_price.iter_rows(named=True):
    if row["town"] not in seen_towns:
        seen_towns.add(row["town"])
        print(f"  {row['town']:<20} S${row['resale_price']:>10,.0f}")
    if len(seen_towns) >= 5:
        break

# --- 9c: Sort by computed expression ---
# Sort by price_per_sqm descending — which transactions were most expensive per sqm?
by_psm = hdb.sort("price_per_sqm", descending=True)
print(f"\n=== Highest Price per sqm ===")
print(by_psm.select("town", "flat_type", "price_per_sqm", "resale_price").head(5))

# --- 9d: Sorting for analysis — find the middle of the market ---
# The 50th percentile transaction (not the same as the median of a column)
n = hdb.height
middle_idx = n // 2
sorted_df = hdb.sort("resale_price")
middle_row = sorted_df.row(middle_idx, named=True)
print(f"\n=== Middle-of-Market Transaction (row {middle_idx:,} of {n:,}) ===")
print(f"  Town: {middle_row['town']}, Type: {middle_row['flat_type']}")
print(
    f"  Price: S${middle_row['resale_price']:,.0f}, Area: {middle_row['floor_area_sqm']:.0f} sqm"
)



### Checkpoint 9


In [ ]:
assert (
    by_price_desc["resale_price"][0] >= by_price_desc["resale_price"][-1]
), "Descending sort: first should be >= last"
assert (
    by_price_asc["resale_price"][0] <= by_price_asc["resale_price"][-1]
), "Ascending sort: first should be <= last"
print("\n✓ Checkpoint 9 passed — sorting working correctly\n")



## TASK 10: Method chaining — building analysis pipelines


In [ ]:
# Method chaining: instead of saving each intermediate step to a variable,
# you attach the next operation with a dot. Polars is designed for this.
# Reading top-to-bottom tells the story of your analysis.

# --- 10a: Recent premium properties pipeline ---
recent_premium = (
    hdb.filter(pl.col("year") >= 2020)
    .filter(pl.col("price_tier").is_in(["premium", "luxury"]))
    .select(
        "transaction_date",
        "town",
        "flat_type",
        "price_per_sqm",
        "price_tier",
        "resale_price",
    )
    .sort("resale_price", descending=True)
)

print(f"\n=== Recent Premium/Luxury Transactions (2020+) ===")
print(f"Count: {recent_premium.height:,}")
print(recent_premium.head(10))

# --- 10b: Town comparison pipeline ---
top_towns = (
    recent_premium.group_by("town")
    .agg(
        pl.len().alias("transaction_count"),
        pl.col("resale_price").median().alias("median_price"),
        pl.col("price_per_sqm").median().alias("median_price_sqm"),
    )
    .sort("transaction_count", descending=True)
)

print(f"\n=== Towns with Most Premium/Luxury Transactions (2020+) ===")
print(top_towns.head(10))

# --- 10c: Year-over-year price growth pipeline ---
annual_median = (
    hdb.group_by("year")
    .agg(
        pl.col("resale_price").median().alias("median_price"),
        pl.col("price_per_sqm").median().alias("median_psm"),
        pl.len().alias("transactions"),
    )
    .sort("year")
)

print(f"\n=== Annual Market Summary ===")
prev_price = None
for row in annual_median.iter_rows(named=True):
    price = row["median_price"]
    if prev_price is not None:
        yoy = (price - prev_price) / prev_price * 100
        arrow = "↑" if yoy > 0 else "↓" if yoy < 0 else "→"
        print(
            f"  {row['year']}: S${price:>10,.0f}  "
            f"{arrow} {yoy:+.1f}%  "
            f"({row['transactions']:,} txns)"
        )
    else:
        print(
            f"  {row['year']}: S${price:>10,.0f}  "
            f"(baseline)  "
            f"({row['transactions']:,} txns)"
        )
    prev_price = price
# INTERPRETATION: The annual summary reveals Singapore's property market cycles.
# Look for: (1) sharp rises after policy relaxation or COVID rebound,
# (2) flat or declining years after cooling measures, (3) transaction
# volume dropping when prices peak — buyers wait for corrections.

# --- 10d: Full analysis pipeline — one chained expression ---
investment_report = (
    hdb.filter(pl.col("year") >= 2022)
    .filter(pl.col("flat_type").is_in(["4 ROOM", "5 ROOM"]))
    .group_by("town")
    .agg(
        pl.len().alias("volume"),
        pl.col("resale_price").median().alias("median_price"),
        pl.col("price_per_sqm").median().alias("median_psm"),
        pl.col("resale_price").std().alias("price_volatility"),
    )
    .with_columns(
        (pl.col("price_volatility") / pl.col("median_price") * 100).alias("cv_pct"),
    )
    .sort("median_psm", descending=True)
)

print(f"\n=== Investment Report: 4/5-Room Flats (2022+) ===")
print(f"{'Town':<22} {'Vol':>6} {'Median':>12} {'PSM':>10} {'CV%':>6}")
print(f"{'─' * 58}")
for row in investment_report.head(15).iter_rows(named=True):
    print(
        f"{row['town']:<22} {row['volume']:>6,} "
        f"S${row['median_price']:>10,.0f} "
        f"S${row['median_psm']:>8,.0f} "
        f"{row['cv_pct']:>5.1f}%"
    )
# INTERPRETATION: Towns at the top have the highest price per sqm — they're
# the most "premium" locations. CV% (coefficient of variation) tells you
# price consistency: low CV means predictable pricing, high CV means wide
# price spreads within the town (a mix of old and new, small and large).
# An investor seeking stable returns wants high volume + low CV.



### Checkpoint 10


In [ ]:
assert recent_premium.height > 0, "recent_premium should have rows"
assert recent_premium.height < hdb.height, "Chained filters should reduce row count"
assert (
    recent_premium["resale_price"][0] >= recent_premium["resale_price"][-1]
), "DataFrame should be sorted descending by resale_price"
assert investment_report.height > 0, "investment_report should have rows"
assert "cv_pct" in investment_report.columns, "cv_pct should be computed"
print("\n✓ Checkpoint 10 passed — method chaining pipelines built correctly\n")



## REFLECTION


✓ Dataset inspection: unique values, ranges, null counts before filtering
  ✓ Boolean filters: pl.col() + comparison operators (==, >, <=, !=)
  ✓ Compound filters: & for AND, | for OR, parentheses for grouping
  ✓ Negation: ~ to invert a condition
  ✓ Set membership: .is_in() for multi-value matching, .is_between() for ranges
  ✓ Null handling: .is_null(), .is_not_null() for missing data checks
  ✓ Column selection: .select() to keep, .drop() to remove
  ✓ Column renaming: .rename({"old": "new"}) for clearer names
  ✓ Feature engineering: .with_columns() + .alias() for new columns
  ✓ Date parsing: str.to_date(), extracting year/month/quarter
  ✓ Conditional logic: pl.when().then().otherwise() for categories
  ✓ Sorting: single-key, multi-key, and by computed expressions
  ✓ Method chaining: building readable analysis pipelines step by step
  ✓ Investment analysis: volume, median, volatility, CV% by district

  NEXT: In Exercise 3, you'll write Python functions and use
  group_by() + agg() to compute statistics for every district
  at once — instead of filtering one town at a time. You'll
  also learn for loops to iterate over results and build reports.


In [ ]:
print("═" * 60)
print("  WHAT YOU'VE MASTERED")
print("═" * 60)
print(
)

